In [ ]:
from fg.variables import Variable, Parameter
from fg.factors import DynamicsFactor, ObservationFactor, PriorFactor
from fg.simulation_config import simulate_wc
from fg.graph import Graph
from fg.gaussian import Gaussian
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import copy
import random

%matplotlib inline

if __name__ == "__main__":
    sigma_obs = 1e-1
    sigma_dynamics = 5e-3
    sigma_prior = 1e1
    iters = 25
    T = 15
    nr = 1
    dt = 0.05
    log_every_n_iters = 1

    schedule = 'Hybrid'

    C = torch.empty((nr, nr)).normal_(0.2, 0.1)
    C.fill_diagonal_(0.)
    
    results_mean = defaultdict(lambda: [])
    results_cov = defaultdict(lambda: [])
    results_E = []
    results_I = []

    gt_config = {
        'T': T,
        'dt': dt,
        'nr': nr,
        'C': C,
        'a': torch.empty((nr,)).uniform_(0.1, 10.),
        'b': torch.empty((nr,)).uniform_(0.1, 10.),
        'c': torch.empty((nr,)).uniform_(0.1, 10.),
        'd': torch.empty((nr,)).uniform_(0.1, 10.),
        'P': torch.empty((nr,)).uniform_(0.1, 1.2),
        'Q': torch.empty((nr,)).uniform_(0.1, 1.2),
        'dyn_noise': 0.01,
        'obs_noise': 0.01,
    }

    config = copy.deepcopy(gt_config)

    E, I = simulate_wc(config)
    plt.plot(E, label='E')
    plt.plot(I, label='I')
    plt.legend()
    plt.show()
    time = torch.arange(0, len(E), 1)
    
    factor_graph = Graph(nr)

    param_list = ['a', 'b', 'c', 'd', 'P', 'Q']

    # -- Construct FG -- #
    # Add our variable and observation factors at each time step
    for t in range(len(time)):
        for r in range(nr):
            factor_graph.var_nodes[f'osc_t{t}_r{r}'] = Variable(
                id       = f'osc_t{t}_r{r}',
                belief   = Gaussian(torch.tensor([[0.1, 0.1]]).T, torch.tensor([[0.2, 0.], [0., 0.2]])),
                graph    = factor_graph, 
                num_vars = 2,
                connected_factors = [(f'osc_t{t}_r{r}', f'osc_t{t+1}_r{r}') if t+1 < len(time) else -1] +  [(f'osc_t{t-1}_r{r}', f'osc_t{t}_r{r}') if t > 0 else -1]
            )
            
            factor_graph.factor_nodes[f'obs_t{t}_r{r}'] = ObservationFactor(
                factor_id = f'obs_t{t}_r{r}', 
                var_id    = f'osc_t{t}_r{r}',
                z         = torch.tensor([[E[t, r], I[t, r]]]).T.float(),
                lmbda_in  = torch.tensor([[sigma_obs ** -2, 0.], [0., sigma_obs ** -2]]),
                graph     = factor_graph
            )

    # Add parameters to each region
    for p in param_list:
        for r in range(nr):
            p_id = f'p({p})_r{r}'

            factor_graph.param_ids.append(p_id)      
            factor_graph.var_nodes[p_id] = Parameter(
                id     = p_id, 
                belief = Gaussian(torch.tensor([[0.]]), torch.tensor([[sigma_prior ** 2.]])),
                graph  = factor_graph,
                connected_factors = [(f'osc_t{t}_r{r}', f'osc_t{t+1}_r{r}') for t in range(len(time)-1)]
            )

            # Add priors to those parameters
            factor_graph.factor_nodes[f'{p_id}_prior'] = PriorFactor(
                factor_id = f'{p_id}_prior',
                var_id = p_id,
                z = torch.tensor([[0.]]).T, 
                lmbda_in = torch.diag(torch.tensor([sigma_prior ** -2])),
                graph = factor_graph
            )

    # Add the dynamics factors between timesteps in every region
    for r in range(nr):
        for t in range(len(time)):
            if t+1 < len(time):
                dyn_id = (f'osc_t{t}_r{r}', f'osc_t{t+1}_r{r}')
                factor_graph.factor_nodes[dyn_id] = DynamicsFactor(
                    Vt_id  = f'osc_t{t}_r{r}',
                    Vtp_id = f'osc_t{t+1}_r{r}',
                    region_id = r,
                    conn = C,
                    lmbda_in = torch.tensor([[sigma_dynamics ** -2., 0.], [0., sigma_dynamics ** -2.]]),
                    factor_id = dyn_id, 
                    graph = factor_graph,
                    connected_params = [f'p({p})_r{r}' for p in param_list]
                )

    # === RUN GBP (Sweep schedule) === #
    for iter in range(iters):
        if iter % log_every_n_iters == 0:
            for r in range(nr):
                for a in param_list:
                    print(factor_graph.var_nodes[f'p({a})_r{r}'])

                    results_mean[a].append(torch.abs((gt_config[a][r] - factor_graph.var_nodes[f'p({a})_r{r}'].belief.mean.item()) / gt_config[a][r]))
                    results_cov[a].append(factor_graph.var_nodes[f'p({a})_r{r}'].belief.cov.item())

                    if factor_graph.var_nodes[f'p({a})_r{r}'].belief.eta.isnan().any(): 
                        print('Found nan, exiting..')
                        exit(0)
            
            # Recreate the signal with the learnt parameters and store the MAE
            for k in param_list:
                for r in range(nr):
                    t = f'p({k})_r{r}'
                    config[k][r] = factor_graph.get_var_belief(t).mean

            config['dyn_noise'] = 0.
            config['obs_noise'] = 0.
            E_rec, I_rec = simulate_wc(config)
            plt.title(f'Iteration {iter}')
            plt.plot(E, 'r', label='GT E')
            plt.plot(E_rec, 'r.', label='Recreated E')
            plt.plot(I, 'b', label='GT I')
            plt.plot(I_rec, 'b.', label='Recreated I')
            plt.legend()
            plt.show()
            results_E.append(np.mean(np.square(E_rec - E)))
            results_I.append(np.mean(np.square(I_rec - I)))

        if iter == 0:
            factor_graph.update_all_observational_factors()

            for i in factor_graph.var_nodes:
                curr = factor_graph.var_nodes[i]
                curr.compute_and_send_messages()

        # Run the different schedule options that we have?
        if schedule == 'Random':
            for i in factor_graph.var_nodes:
                if i.__class__.__name__ == 'Variable':
                    curr = factor_graph.var_nodes[i]
                    curr.compute_and_send_messages()
            
            for i in factor_graph.factor_nodes:
                curr = factor_graph.factor_nodes[i]
                curr.compute_and_send_messages() 

            factor_graph.update_params()
            for j in factor_graph.param_ids: factor_graph.factor_nodes[f'{j}_prior'].belief = factor_graph.var_nodes[j].belief
            factor_graph.update_all_observational_factors()

        elif schedule == 'Sweep':
            # Right Pass
            for t in range(len(time)-1):
                for r in range(nr):
                    curr = factor_graph.var_nodes[f'osc_t{t}_r{r}']
                    curr.compute_and_send_messages()

                    if t+1 == len(time): continue
                    
                    factor_graph.factor_nodes[(f'osc_t{t}_r{r}', f'osc_t{t+1}_r{r}')].compute_and_send_messages()
                
            factor_graph.update_params() 
            for j in factor_graph.param_ids: factor_graph.factor_nodes[f'{j}_prior'].belief = factor_graph.var_nodes[j].belief
            factor_graph.update_all_observational_factors()

            # Left Pass
            for t in range(len(time)-2, 0, -1):
                for r in range(nr):
                    curr = factor_graph.var_nodes[f'osc_t{t}_r{r}']
                    curr.compute_and_send_messages()

                    if t-1 == 0: continue

                    # Update dynamical factor
                    factor_graph.factor_nodes[(f'osc_t{t-1}_r{r}', f'osc_t{t}_r{r}')].compute_and_send_messages()
                
            factor_graph.update_params()
            for j in factor_graph.param_ids: factor_graph.factor_nodes[f'{j}_prior'].belief = factor_graph.var_nodes[j].belief
            factor_graph.update_all_observational_factors()
        
        elif schedule == 'Hybrid':
            alpha_left = 0
            for alpha in range(15, len(time), 15):
                for _ in range(30):
                    
                    var_nodes = [(t,r) for t in range(alpha) for r in range(nr)]
                    random.shuffle(var_nodes)

                    for t, r in var_nodes:
                        curr = factor_graph.var_nodes[f'osc_t{t}_r{r}']
                        curr.compute_and_send_messages()

                    factor_nodes = [(t,r) for t in range(alpha-1) for r in range(nr)]
                    random.shuffle(factor_nodes) 

                    for t, r in factor_nodes:
                        factor_graph.factor_nodes[(f'osc_t{t}_r{r}', f'osc_t{t+1}_r{r}')].compute_and_send_messages()    

                    factor_graph.update_params()
                for j in factor_graph.param_ids: factor_graph.factor_nodes[f'{j}_prior'].belief = factor_graph.var_nodes[j].belief
                factor_graph.update_all_observational_factors()
                alpha_left += 15
                   

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rc('text', usetex=True)
plt.rc('font', family='serif')

def plot_errors_separate(error_dict):
    # Group data by parameter
    param_data = defaultdict(list)
    for (run, param), values in error_dict.items():
        param_data[param].append(values)
    
    # Calculate mean, min, and max for each parameter
    param_stats = {}
    for param, data_list in param_data.items():
        data_array = np.array(data_list)
        param_stats[param] = {
            'mean': np.mean(data_array, axis=0),
            'min': np.min(data_array, axis=0),
            'max': np.max(data_array, axis=0)
        }
    
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Percentage Error Over Iterations for Each Parameter')
    
    for (i, j), (param, stats) in zip(np.ndindex(2, 3), param_stats.items()):
        ax = axes[i, j]
        x = np.arange(len(stats['mean']))
        ax.plot(x, stats['mean'], label='Mean')
        ax.fill_between(x, stats['min'], stats['max'], alpha=0.2, label='Min-Max Range')
        
        # Adjust x-axis ticks
        ax.set_ylim(bottom = -1, top = 1)
        ax.set_xticks(np.arange(0, 31, 5), np.arange(0, 301, 50))
        
        ax.set_xlabel('Iterations')
        ax.set_ylabel('Percentage Error')
        ax.set_title(f'Parameter ${param}$')
        ax.legend()
        ax.grid(True)
    
    plt.tight_layout()
    # plt.savefig('abcdPQ_params_WC_10runs.pdf', dpi=600, bbox_inches='tight')
    plt.show()

plot_errors_separate(results_mean)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_twin_results(results_E, results_I):
    # Calculate statistics for both datasets
    def calculate_stats(data):
        values = np.array(list(data.values()))
        return {
            'mean': np.mean(values, axis=0),
            'min': np.min(values, axis=0),
            'max': np.max(values, axis=0)
        }
    
    stats_E = calculate_stats(results_E)
    stats_I = calculate_stats(results_I)
    
    # Create the plot
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()
    
    # Plot data for results_E
    x = np.arange(len(stats_E['mean']))
    ax1.plot(x, stats_E['mean'], 'b-', label='E Mean')
    ax1.fill_between(x, stats_E['min'], stats_E['max'], alpha=0.2, color='b')
    
    # Plot data for results_I
    ax2.plot(x, stats_I['mean'], 'r-', label='I Mean')
    ax2.fill_between(x, stats_I['min'], stats_I['max'], alpha=0.2, color='r')
    
    # Set labels and title
    ax1.set_xlabel('Iterations')
    ax1.set_ylabel('E Values', color='b')
    ax2.set_ylabel('I Values', color='r')
    plt.title('Mean Values with Min-Max Range for E and I Results')
    
    # Adjust x-axis ticks
    x_ticks = np.arange(0, len(x) * 10, 10)
    plt.xticks(np.arange(len(x)), x_ticks)
    
    # Add legends
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    
    # Add grid
    ax1.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_twin_results(results_E, results_I)

In [ ]:
def compute_nn_size(layer_sizes):
    s = 0
    for l in range(len(layer_sizes)-1):
        s += layer_sizes[l] * layer_sizes[l+1] +layer_sizes[l+1]  

    return s

compute_nn_size([2,3,3,2])

In [ ]:
from fg.variables import Variable, Parameter
from fg.factors import PriorFactor
from torch.nn.functional import linear, relu
import torch
from fg.graph import Graph
from fg.gaussian import Gaussian
from collections import defaultdict
import matplotlib.pyplot as plt

class MLPFactor:
    def __init__(self, factor_id, z, lmbda_in, graph : Graph, huber = False):
        self.factor_id = factor_id

        self.lmbda_in = lmbda_in
        self.graph = graph
        self.huber = huber

        self._connected_vars = ['xt'] + [f'x{i}' for i in range(d)] + [f'p{i}' for i in range(d)] 

        self.z = z

        self.inbox = {}

        # Used for message damping, see Ortiz (2023) 3.4.6
        self._prev_messages = {}

    def _h_fn(self, Nt, *input):        
        X = torch.cat(input[:d]).T
        W = torch.cat(input[d:2*d]).T

        act = linear(X, W)

        return torch.abs(Nt - act)

    def linearise(self) -> Gaussian:
        '''
        Returns the linearised Gaussian factor based on equations 2.46 and 2.47 in Ortiz (2023)
        '''
        connected_variables = []
        for i in self._connected_vars:
            mean = self.graph.get_var_belief(i).mean.detach().clone()
            if mean.numel() > 1: #nD beliefs
                for j in range(mean.numel()):
                    connected_variables.append(mean[j].reshape(1, 1))
            else: #1D beliefs
                connected_variables.append(mean.reshape(1, 1))

        Nt = connected_variables[0]
        weights = connected_variables[1:]

        self.h = self._h_fn(Nt, *weights)

        J = torch.concat(torch.autograd.functional.jacobian(self._h_fn, (Nt, *weights)), 0)[..., 0, 0].T
        x0 = torch.concat([v for v in connected_variables], dim=0)

        eta = (J.T @ self.lmbda_in) @ (-self.h.T + J @ x0)
        lmbda = (J.T @ self.lmbda_in) @ J 

        return Gaussian.from_canonical(eta.detach(), lmbda.detach())

    def _compute_message_to_i(self, i, beta = 0.5) -> Gaussian:
        '''
        Compute message to variable at index i in `self._vars`,
        All of this is eqn 8 from 'Learning in Deep Factor Graphs with Gaussian Belief Propagation'
        '''
        linearised_factor = self.linearise()

        product = Gaussian.zeros_like(linearised_factor)

        # Build our message product by adding corresponding eta and lambda
        # in product
        k = 0
        for j, id in enumerate(self._connected_vars):
            if j != i:
                in_msg = self.inbox.get(id, Gaussian.from_canonical(torch.tensor([0.]), \
                    torch.tensor([0.])))

                offset = in_msg.eta.numel()
                product.eta[k : k+offset] += in_msg.eta
                product.lmbda[k : k+offset, k : k+offset] += in_msg.lmbda

                k += offset
            else:
                k += self.graph.var_nodes[self._connected_vars[i]].num_vars
                # k += 2 if i in [0,1] else 1

        factor_product = linearised_factor * product

        start_idx = 0
        for k in range(i):
            start_idx += self.graph.var_nodes[self._connected_vars[k]].num_vars

        idx_to_marginalise = list(range(start_idx, start_idx + self.graph.var_nodes[self._connected_vars[i]].num_vars))

        marginal = factor_product.marginalise(idx_to_marginalise)

        kR = self.compute_huber() if self.huber else 1.
        marginal *= kR

        prev_msg = self._prev_messages.get(i, Gaussian.zeros_like(marginal))
        damped_factor = (marginal * beta) * (prev_msg * (1 - beta))

        # Store previous message
        self._prev_messages[i] = damped_factor

        return damped_factor

    def compute_and_send_messages(self) -> None:
        for i, var_id in enumerate(self._connected_vars):
            msg = self._compute_message_to_i(i)
            self.graph.send_msg_to_variable(self.factor_id, var_id, msg)

    def __str__(self):
        return f'MLP: Var: {self.var_id}' 


# Set up problem
n = 150
d = 6
start_idx = 0

gt = torch.rand(1, d)*10
X = torch.rand(n, d)
y = linear(X, gt)

factor_graph = Graph(1)

factor_graph.factor_nodes['mlp'] = MLPFactor(
    factor_id = 'mlp',
    lmbda_in = torch.tensor([[0.1 ** -1.]]),
    graph = factor_graph,
    z = torch.tensor([[0.]])
)

factor_graph.var_nodes['xt'] = Variable(
    id = 'xt',
    belief = Gaussian(torch.tensor([[0.]]), torch.tensor([[0.2]])),
    num_vars = 1,
    graph = factor_graph,
    connected_factors = ['mlp']
)

factor_graph.factor_nodes['xt_prior'] = PriorFactor(
    factor_id = 'xt_prior',
    var_id = 'xt',
    z = torch.tensor([[y[start_idx]]]),
    lmbda_in = torch.tensor([[0.1 ** -2.]]),
    graph = factor_graph
)

for i in range(d):
    factor_graph.var_nodes[f'x{i}'] = Variable(
        id = f'x{i}',
        belief = Gaussian(torch.tensor([[0.]]).T, torch.tensor([[1e-1]])),
        num_vars = 1,
        connected_factors = ['mlp'],
        graph = factor_graph
    )

    factor_graph.factor_nodes[f'x{i}_obs'] = PriorFactor(
        factor_id = f'x{i}_obs',
        var_id = f'x{i}',
        z = torch.tensor([[X[start_idx,i].item()]]),
        lmbda_in = torch.tensor([[1e-1 ** -2]]),
        graph = factor_graph
    )

    factor_graph.param_ids.append(f'p{i}')
    factor_graph.var_nodes[f'p{i}'] = Parameter(
        id     = f'p{i}', 
        belief = Gaussian(torch.tensor([[0.]]), torch.tensor([[10 ** 2.]])),
        graph  = factor_graph,
        connected_factors = ['mlp']
    )

    # Add priors to those parameters
    factor_graph.factor_nodes[f'p{i}_prior'] = PriorFactor(
        factor_id = f'p{i}_prior',
        var_id = f'p{i}',
        z = torch.tensor([[0.]]).T, 
        lmbda_in = torch.diag(torch.tensor([10 ** -2.])),
        graph = factor_graph
    )

factor_graph.update_all_observational_factors()

print(gt)

In [ ]:
res_mean = defaultdict(lambda: [])
res_cov = defaultdict(lambda: [])

for i in range(n):
    for j in range(d):
        factor_graph.factor_nodes[f'x{j}_obs'].set_z(torch.tensor([[X[i,j].item()]]))
    factor_graph.factor_nodes['xt_prior'].set_z(torch.tensor([[y[i].item()]]))

    factor_graph.update_all_observational_factors()

    for _ in range(100):
        for _, j in factor_graph.var_nodes.items(): j.compute_and_send_messages()

        factor_graph.factor_nodes['mlp'].compute_and_send_messages()

        for _, j in factor_graph.var_nodes.items(): j.update_belief()
    
    # Use the param belief as new priors
    for j in range(d):
        factor_graph.factor_nodes[f'p{j}_prior'].belief = factor_graph.var_nodes[f'p{j}'].belief

    print(f'--- {i} ---')
    for j in range(d):
        res_mean[j].append(factor_graph.var_nodes[f'p{j}'].mean.item())
        res_cov[j].append(factor_graph.var_nodes[f'p{j}'].cov.item())
        print(factor_graph.var_nodes[f'p{j}'].mean, factor_graph.var_nodes[f'p{j}'].cov)

In [ ]:
import numpy as np
plt.plot(np.abs(gt[0,0].item() - np.array(res_mean[0])))
plt.ylim(bottom=0)